In [1]:
# ablate_runner.py
# ============================================================
# Catch & Climb LSTM Actor-Critic (POMDP CartPole partial obs)
# 2차 실험(아블리게이션): 각 variant를 5000 episodes씩 공정 비교
#
# Variants (요청 반영):
# - FULL      : KL(anchor) + SIL(spike) + extra updates ON
# - NoSpike   : SIL OFF (spike 학습 제거)
# - NoCollapse: KL(anchor) OFF (collapse 학습 제거)
# - NoEventUpd: 이벤트 발생해도 extra updates OFF
# - Vanilla   : KL/SIL/extra 전부 OFF (기본 REINFORCE+critic)
#
# Outputs:
# - 터미널 요약표
# - ablation_raw.csv (seed별)
# - ablation_summary.csv (variant 평균)
# - (옵션) TensorBoard logs: runs_ablation/
# ============================================================

import os
import time
import random
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Bernoulli

try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_OK = True
except Exception:
    TENSORBOARD_OK = False


# =========================
# Base hyperparameters
# =========================
GAMMA = 0.99

ACTOR_HIDDEN = 256
CRITIC_HIDDEN = 256
LSTM_LAYERS = 1
LSTM_DROPOUT = 0.1

LR_ACTOR = 2e-4
LR_CRITIC = 2e-4
CLIP_NORM = 0.5

ENT_COEF = 0.01
USE_HUBER_VALUE_LOSS = True

# Catch & Climb
BASELINE_BETA = 0.90
EPS = 1e-6

AUX_KL_COEF = 0.5
AUX_KL_MAX = 5.0

SIL_COEF = 0.5
SIL_MAX = 5.0

EXTRA_UPDATES_ON_EVENT = 2
EXTRA_UPDATES_CAP = 6

# Env / solve criterion
ENV_ID = "CartPole-v1"
MAX_STEPS_PER_EP = 500
SOLVE_AVG100 = 475.0  # CartPole-v1 typical "solved"


# =========================
# Repro / device
# =========================
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device =", device)


# =========================
# Networks
# =========================
class PolicyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 64)
        self.lstm = nn.LSTM(
            64, ACTOR_HIDDEN,
            num_layers=LSTM_LAYERS,
            dropout=(LSTM_DROPOUT if LSTM_LAYERS >= 2 else 0.0),
            batch_first=True
        )
        self.fc2 = nn.Linear(ACTOR_HIDDEN, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden):
        x = self.relu(self.fc1(x))
        x, hidden = self.lstm(x, hidden)
        x = self.relu(x)
        x = self.sigmoid(self.fc2(x))  # (B,T,1)
        return x, hidden

    def select_action(self, state, hidden):
        with torch.no_grad():
            prob, hidden = self.forward(state, hidden)
            b = Bernoulli(prob)
            action = b.sample()
        return int(action.item()), hidden


class ValueNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 64)
        self.lstm = nn.LSTM(
            64, CRITIC_HIDDEN,
            num_layers=LSTM_LAYERS,
            dropout=(LSTM_DROPOUT if LSTM_LAYERS >= 2 else 0.0),
            batch_first=True
        )
        self.fc2 = nn.Linear(CRITIC_HIDDEN, 1)
        self.relu = nn.ReLU()

    def forward(self, x, hidden):
        x = self.relu(self.fc1(x))
        x, hidden = self.lstm(x, hidden)
        x = self.relu(x)
        x = self.fc2(x)
        return x, hidden


# =========================
# Helpers
# =========================
def obs_to_partial(obs):
    return np.array([obs[0], obs[2]], dtype=np.float32)


def bernoulli_kl(p, q, eps=1e-6):
    p = torch.clamp(p, eps, 1.0 - eps)
    q = torch.clamp(q, eps, 1.0 - eps)
    return p * torch.log(p / q) + (1.0 - p) * torch.log((1.0 - p) / (1.0 - q))


@torch.no_grad()
def forward_policy_probs(policy_net, states_tensor):
    a_hx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=states_tensor.device)
    a_cx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=states_tensor.device)
    prob, _ = policy_net(states_tensor, (a_hx, a_cx))
    return prob.squeeze(0)  # (T,1)


def compute_climb_signals(ep_reward, baseline_prev):
    denom = abs(baseline_prev) + 1.0
    collapse_raw = max(0.0, baseline_prev - ep_reward)
    spike_raw = max(0.0, ep_reward - baseline_prev)

    collapse_int = collapse_raw / (denom + EPS)
    spike_int = spike_raw / (denom + EPS)

    is_collapse = 1.0 if ep_reward < baseline_prev else 0.0
    is_spike = 1.0 if ep_reward > baseline_prev else 0.0
    return is_collapse, is_spike, collapse_int, spike_int


def discounted_returns(rewards: List[float], gamma: float) -> np.ndarray:
    out = np.zeros(len(rewards), dtype=np.float32)
    R = 0.0
    for i in reversed(range(len(rewards))):
        R = gamma * R + rewards[i]
        out[i] = R
    return out


def compute_metrics(reward_list: List[float], solve_thr_avg100: float = SOLVE_AVG100) -> Dict[str, float]:
    r = np.asarray(reward_list, dtype=np.float32)
    auc_mean = float(r.mean())

    tail = r[-100:] if len(r) >= 100 else r
    last100_mean = float(tail.mean())
    last100_std = float(tail.std(ddof=0))

    first_solve = -1
    if len(r) >= 100:
        win = np.convolve(r, np.ones(100, dtype=np.float32) / 100.0, mode="valid")
        idx = np.where(win >= solve_thr_avg100)[0]
        if len(idx) > 0:
            first_solve = int(idx[0] + 100)

    eps = 1e-6
    stability = 1.0 - (last100_std / (abs(last100_mean) + eps))
    stability = float(max(-5.0, min(1.0, stability)))

    return {
        "auc_mean": auc_mean,
        "last100_mean": last100_mean,
        "last100_std": last100_std,
        "first_solve_ep(avg100>=thr)": float(first_solve),
        "stability_last100": stability,
    }


# =========================
# Ablation config
# =========================
@dataclass
class Variant:
    name: str
    use_kl_anchor: bool = True
    use_sil: bool = True
    use_extra_updates_on_event: bool = True
    use_entropy: bool = True
    use_adv_norm: bool = True


def get_variants() -> List[Variant]:
    # 요청한 "그 아블리게이션" 구성으로 고정
    return [
        Variant("FULL",      use_kl_anchor=True,  use_sil=True,  use_extra_updates_on_event=True,  use_entropy=True, use_adv_norm=True),
        Variant("NoSpike",   use_kl_anchor=True,  use_sil=False, use_extra_updates_on_event=True,  use_entropy=True, use_adv_norm=True),
        Variant("NoCollapse",use_kl_anchor=False, use_sil=True,  use_extra_updates_on_event=True,  use_entropy=True, use_adv_norm=True),
        Variant("NoEventUpd",use_kl_anchor=True,  use_sil=True,  use_extra_updates_on_event=False, use_entropy=True, use_adv_norm=True),
        Variant("Vanilla",   use_kl_anchor=False, use_sil=False, use_extra_updates_on_event=False, use_entropy=True, use_adv_norm=True),
    ]


# =========================
# One run (one seed, one variant)
# =========================
def run_variant(
    variant: Variant,
    episodes: int = 5000,
    seed: int = 0,
    log_dir_root: Optional[str] = "./runs_ablation",
    save_policy_every: int = 1000,
    verbose_every: int = 100,
) -> Dict[str, object]:
    set_global_seed(seed)

    env = gym.make(ENV_ID)

    policy = PolicyNetwork().to(device)
    value = ValueNetwork().to(device)

    optim = torch.optim.Adam(policy.parameters(), lr=LR_ACTOR)
    value_optim = torch.optim.Adam(value.parameters(), lr=LR_CRITIC)

    # Anchor snapshot
    anchor_policy = PolicyNetwork().to(device)
    anchor_policy.load_state_dict(policy.state_dict())
    anchor_policy.eval()

    best_reward = -1e9
    baseline = 0.0

    # TensorBoard
    writer = None
    run_tag = f"{variant.name}_seed{seed}_{int(time.time())}"
    if log_dir_root is not None and TENSORBOARD_OK:
        os.makedirs(log_dir_root, exist_ok=True)
        writer = SummaryWriter(os.path.join(log_dir_root, run_tag))

    ep_rewards: List[float] = []
    collapse_count = 0
    spike_count = 0

    for epoch in range(episodes):
        obs, info = env.reset(seed=seed * 100000 + epoch)
        state = obs_to_partial(obs)
        episode_reward = 0.0

        a_hx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
        a_cx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)

        rewards, actions, states = [], [], []

        for t in range(MAX_STEPS_PER_EP):
            states.append(state.copy())

            state_t = torch.tensor(state, dtype=torch.float32, device=device).view(1, 1, 2)
            action, (a_hx, a_cx) = policy.select_action(state_t, (a_hx, a_cx))
            actions.append(action)

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            next_state = obs_to_partial(next_obs)
            episode_reward += float(reward)

            rewards.append(float(reward))
            state = next_state

            if done:
                break

        ep_rewards.append(float(episode_reward))

        # returns
        returns = discounted_returns(rewards, GAMMA)
        mean, std = returns.mean(), returns.std()
        std = std if std > 1e-8 else 1.0
        returns_norm = (returns - mean) / std

        states_tensor = torch.tensor(np.array(states), dtype=torch.float32, device=device).unsqueeze(0)  # (1,T,2)
        actions_tensor = torch.tensor(np.array(actions), dtype=torch.float32, device=device).view(-1, 1)  # (T,1)
        returns_tensor = torch.tensor(returns_norm, dtype=torch.float32, device=device).view(-1, 1)       # (T,1)

        # Catch & Climb signals
        baseline_prev = float(baseline)
        is_collapse, is_spike, collapse_int, spike_int = compute_climb_signals(
            ep_reward=episode_reward, baseline_prev=baseline_prev
        )
        baseline = BASELINE_BETA * baseline + (1.0 - BASELINE_BETA) * float(episode_reward)

        if is_collapse:
            collapse_count += 1
        if is_spike:
            spike_count += 1

        # dynamic aux weights
        kl_w = 0.0
        sil_w = 0.0
        if variant.use_kl_anchor and is_collapse:
            kl_w = min(AUX_KL_MAX, AUX_KL_COEF * (1.0 + 5.0 * collapse_int))
        if variant.use_sil and is_spike:
            sil_w = min(SIL_MAX, SIL_COEF * (1.0 + 5.0 * spike_int))

        # extra updates
        extra_updates = 0
        if variant.use_extra_updates_on_event and (is_collapse or is_spike):
            extra_updates = int(min(EXTRA_UPDATES_CAP, EXTRA_UPDATES_ON_EVENT))

        # critic baseline & advantage
        with torch.no_grad():
            c_hx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
            c_cx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
            v, _ = value(states_tensor, (c_hx, c_cx))
            v = v.squeeze(0)  # (T,1)

            advantage = returns_tensor - v
            if variant.use_adv_norm:
                advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)

        # anchor probs once
        anchor_prob = None
        if variant.use_kl_anchor:
            with torch.no_grad():
                anchor_prob = forward_policy_probs(anchor_policy, states_tensor)  # (T,1)

        # actor step
        def actor_step():
            a_hx0 = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
            a_cx0 = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
            prob, _ = policy(states_tensor, (a_hx0, a_cx0))
            prob = prob.squeeze(0)  # (T,1)

            dist = Bernoulli(prob)
            log_prob = dist.log_prob(actions_tensor)
            entropy = dist.entropy().mean()

            ent_term = (ENT_COEF * entropy) if variant.use_entropy else 0.0
            base_loss = -(log_prob * advantage.detach()).mean() - ent_term

            kl_loss = 0.0
            if kl_w > 0.0 and anchor_prob is not None:
                kl = bernoulli_kl(prob, anchor_prob).mean()
                kl_loss = float(kl_w) * kl

            sil_loss = 0.0
            if sil_w > 0.0:
                pos_adv = torch.clamp(advantage.detach(), min=0.0)
                sil_loss = float(sil_w) * (-(log_prob * pos_adv).mean())

            loss = base_loss + kl_loss + sil_loss

            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), CLIP_NORM)
            optim.step()

            def to_float(x):
                return float(x.item()) if hasattr(x, "item") else float(x)

            return (
                to_float(loss),
                to_float(base_loss),
                to_float(kl_loss),
                to_float(sil_loss),
                to_float(entropy),
            )

        actor_total, actor_base, actor_kl, actor_sil, entropy_v = actor_step()
        for _ in range(extra_updates):
            actor_total, actor_base, actor_kl, actor_sil, entropy_v = actor_step()

        # critic update
        c_hx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
        c_cx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
        v_pred, _ = value(states_tensor, (c_hx, c_cx))
        v_pred = v_pred.squeeze(0)

        if USE_HUBER_VALUE_LOSS:
            value_loss = F.smooth_l1_loss(v_pred, returns_tensor)
        else:
            value_loss = F.mse_loss(v_pred, returns_tensor)

        value_optim.zero_grad()
        value_loss.backward()
        torch.nn.utils.clip_grad_norm_(value.parameters(), CLIP_NORM)
        value_optim.step()

        # anchor update (best policy)
        if variant.use_kl_anchor and episode_reward > best_reward:
            best_reward = float(episode_reward)
            anchor_policy.load_state_dict(policy.state_dict())
            anchor_policy.eval()

        # logging
        if writer is not None:
            writer.add_scalar("episode_reward", episode_reward, epoch)
            writer.add_scalar("baseline/ema", baseline, epoch)
            writer.add_scalar("catch/is_collapse", is_collapse, epoch)
            writer.add_scalar("catch/is_spike", is_spike, epoch)
            writer.add_scalar("catch/collapse_int", collapse_int, epoch)
            writer.add_scalar("catch/spike_int", spike_int, epoch)
            writer.add_scalar("catch/kl_weight", kl_w, epoch)
            writer.add_scalar("catch/sil_weight", sil_w, epoch)
            writer.add_scalar("catch/extra_updates", extra_updates, epoch)
            writer.add_scalar("loss/actor_total", actor_total, epoch)
            writer.add_scalar("loss/actor_base", actor_base, epoch)
            writer.add_scalar("loss/actor_kl", actor_kl, epoch)
            writer.add_scalar("loss/actor_sil", actor_sil, epoch)
            writer.add_scalar("loss/value", float(value_loss.item()), epoch)
            writer.add_scalar("stats/entropy", entropy_v, epoch)
            writer.add_scalar("best_reward", best_reward, epoch)

        if (epoch % verbose_every) == 0:
            tag = "COLLAPSE" if is_collapse else ("SPIKE" if is_spike else "normal")
            print(
                f"[{variant.name} | seed={seed}] ep {epoch:05d} | "
                f"R {episode_reward:6.1f} | ema {baseline:7.2f} | {tag} | extra={extra_updates}"
            )

        if save_policy_every > 0 and (epoch + 1) % save_policy_every == 0:
            torch.save(policy.state_dict(), f"policy_{variant.name}_seed{seed}_ep{epoch+1}.pt")

    if writer is not None:
        writer.close()
    env.close()

    metrics = compute_metrics(ep_rewards, SOLVE_AVG100)
    out = {
        "variant": variant.name,
        "seed": seed,
        "episodes": episodes,
        "collapse_count": float(collapse_count),
        "spike_count": float(spike_count),
        **metrics,
    }
    return out


# =========================
# Summary utilities
# =========================
def format_table(rows: List[Dict[str, object]], cols: List[str]) -> str:
    col_widths = {c: max(len(c), max(len(f"{r.get(c, '')}") for r in rows)) for c in cols}
    header = " | ".join(c.ljust(col_widths[c]) for c in cols)
    sep = "-+-".join("-" * col_widths[c] for c in cols)
    lines = [header, sep]
    for r in rows:
        lines.append(" | ".join(f"{r.get(c, '')}".ljust(col_widths[c]) for c in cols))
    return "\n".join(lines)


def aggregate_across_seeds(results: List[Dict[str, object]]) -> List[Dict[str, object]]:
    by: Dict[str, List[Dict[str, object]]] = {}
    for r in results:
        by.setdefault(str(r["variant"]), []).append(r)

    keys = [
        "auc_mean",
        "last100_mean",
        "last100_std",
        "first_solve_ep(avg100>=thr)",
        "stability_last100",
        "collapse_count",
        "spike_count",
    ]

    agg = []
    for v, rs in by.items():
        out = {"variant": v, "n_seeds": len(rs)}
        for k in keys:
            vals = [float(x[k]) for x in rs]
            out[k] = float(np.mean(vals))
        agg.append(out)

    agg.sort(key=lambda x: x["last100_mean"], reverse=True)
    return agg


def save_csv(path: str, rows: List[Dict[str, object]]):
    import csv
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    cols = sorted({k for r in rows for k in r.keys()})
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in rows:
            w.writerow(r)


# =========================
# Main
# =========================
def main():
    EPISODES_PER_VARIANT = 5000     # 2차 실험: 각 5000
    SEEDS = [0]                    # 평균 내고 싶으면: [0,1,2]
    LOG_ROOT = "./runs_ablation"   # TensorBoard 끄려면 None

    variants = get_variants()

    all_results: List[Dict[str, object]] = []
    for v in variants:
        for s in SEEDS:
            print(f"\n=== RUN: {v.name} (seed={s}) episodes={EPISODES_PER_VARIANT} device={device} ===")
            res = run_variant(
                v,
                episodes=EPISODES_PER_VARIANT,
                seed=s,
                log_dir_root=LOG_ROOT,
                save_policy_every=1000,
                verbose_every=100,
            )
            all_results.append(res)

    agg = aggregate_across_seeds(all_results)

    cols = [
        "variant", "n_seeds",
        "auc_mean", "last100_mean", "last100_std",
        "first_solve_ep(avg100>=thr)",
        "stability_last100",
        "collapse_count", "spike_count",
    ]

    pretty = []
    for r in agg:
        pr = dict(r)
        for k in ["auc_mean", "last100_mean", "last100_std", "stability_last100"]:
            pr[k] = f"{pr[k]:.3f}"
        pr["first_solve_ep(avg100>=thr)"] = f"{int(pr['first_solve_ep(avg100>=thr)'])}"
        pr["collapse_count"] = f"{pr['collapse_count']:.1f}"
        pr["spike_count"] = f"{pr['spike_count']:.1f}"
        pretty.append(pr)

    print("\n\n========== ABLATION SUMMARY (mean across seeds) ==========")
    print(format_table(pretty, cols))

    save_csv("./ablation_raw.csv", all_results)
    save_csv("./ablation_summary.csv", agg)
    print("\nSaved: ablation_raw.csv, ablation_summary.csv")

    if LOG_ROOT is not None and TENSORBOARD_OK:
        print(f"TensorBoard:  tensorboard --logdir {LOG_ROOT}")


if __name__ == "__main__":
    main()


device = cpu

=== RUN: FULL (seed=0) episodes=5000 device=cpu ===
[FULL | seed=0] ep 00000 | R   17.0 | ema    1.70 | SPIKE | extra=2
[FULL | seed=0] ep 00100 | R   33.0 | ema   25.66 | SPIKE | extra=2
[FULL | seed=0] ep 00200 | R   11.0 | ema   19.99 | COLLAPSE | extra=2
[FULL | seed=0] ep 00300 | R   43.0 | ema   29.31 | SPIKE | extra=2
[FULL | seed=0] ep 00400 | R   22.0 | ema   37.27 | COLLAPSE | extra=2
[FULL | seed=0] ep 00500 | R   14.0 | ema   27.29 | COLLAPSE | extra=2
[FULL | seed=0] ep 00600 | R   29.0 | ema   30.20 | COLLAPSE | extra=2
[FULL | seed=0] ep 00700 | R   39.0 | ema   31.32 | SPIKE | extra=2
[FULL | seed=0] ep 00800 | R   19.0 | ema   30.80 | COLLAPSE | extra=2
[FULL | seed=0] ep 00900 | R   22.0 | ema   36.65 | COLLAPSE | extra=2
[FULL | seed=0] ep 01000 | R   38.0 | ema   39.19 | COLLAPSE | extra=2
[FULL | seed=0] ep 01100 | R   15.0 | ema   37.63 | COLLAPSE | extra=2
[FULL | seed=0] ep 01200 | R   12.0 | ema   56.33 | COLLAPSE | extra=2
[FULL | seed=0] ep 0130